In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    count,
    when,
    sum as spark_sum,
    round as spark_round,
    mean,
    stddev,
    min as spark_min,
    max as spark_max,
    isnan
)

spark = (
    SparkSession.builder
    .appName("EDA Train Ready Hotel Booking Data")
    .master("spark://100.127.25.114:7077")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/11 10:50:36 WARN Utils: Your hostname, Evelyn-Nguyen.local, resolves to a loopback address: 127.0.0.1; using 192.168.22.100 instead (on interface en0)
26/06/11 10:50:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/11 10:50:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
train_csv_path = "hdfs://100.127.25.114:9000/data/hotel_booking_training/cleaned_hotel_bookings"

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(train_csv_path)
)

26/06/11 10:50:40 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: hdfs://100.127.25.114:9000/data/hotel_booking_training/cleaned_hotel_bookings.
java.io.FileNotFoundException: File does not exist: hdfs://100.127.25.114:9000/data/hotel_booking_training/cleaned_hotel_bookings
	at org.apache.hadoop.hdfs.DistributedFileSystem$29.doCall(DistributedFileSystem.java:1842)
	at org.apache.hadoop.hdfs.DistributedFileSystem$29.doCall(DistributedFileSystem.java:1835)
	at org.apache.hadoop.fs.FileSystemLinkResolver.resolve(FileSystemLinkResolver.java:81)
	at org.apache.hadoop.hdfs.DistributedFileSystem.getFileStatus(DistributedFileSystem.java:1850)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysi

AnalysisException: [PATH_NOT_FOUND] Path does not exist: hdfs://100.127.25.114:9000/data/hotel_booking_training/cleaned_hotel_bookings. SQLSTATE: 42K03

26/06/11 13:03:11 WARN HeartbeatReceiver: Removing executor 0 with no recent heartbeats: 925575 ms exceeds timeout 120000 ms
26/06/11 13:03:11 WARN HeartbeatReceiver: Removing executor 1 with no recent heartbeats: 925190 ms exceeds timeout 120000 ms
26/06/11 13:03:12 ERROR Inbox: Ignoring error
java.lang.AssertionError: assertion failed: BlockManager re-registration shouldn't succeed when the executor is lost
	at scala.Predef$.assert(Predef.scala:279)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$register(BlockManagerMasterEndpoint.scala:748)
	at org.apache.spark.storage.BlockManagerMasterEndpoint$$anonfun$receiveAndReply$1.applyOrElse(BlockManagerMasterEndpoint.scala:141)
	at org.apache.spark.rpc.netty.Inbox.$anonfun$process$1(Inbox.scala:104)
	at org.apache.spark.rpc.netty.Inbox.safelyCall(Inbox.scala:216)
	at org.apache.spark.rpc.netty.Inbox.process(Inbox.scala:101)
	at org.apache.spark.rpc.netty.MessageLoop.org$apache$s

Xem size & shape

In [8]:
df.printSchema()

total_rows = df.count()
total_cols = len(df.columns)

print("Total rows:", total_rows)
print("Total columns:", total_cols)

root
 |-- label: integer (nullable = true)
 |-- lead_time: integer (nullable = true)
 |-- stays_in_weekend_nights: integer (nullable = true)
 |-- stays_in_week_nights: integer (nullable = true)
 |-- total_nights: integer (nullable = true)
 |-- adults: integer (nullable = true)
 |-- children: integer (nullable = true)
 |-- babies: integer (nullable = true)
 |-- total_guests: integer (nullable = true)
 |-- booking_changes: integer (nullable = true)
 |-- days_in_waiting_list: integer (nullable = true)
 |-- adr: double (nullable = true)
 |-- is_repeated_guest: integer (nullable = true)
 |-- previous_cancellations: integer (nullable = true)
 |-- previous_bookings_not_canceled: integer (nullable = true)
 |-- agent: integer (nullable = true)
 |-- company: integer (nullable = true)
 |-- required_car_parking_spaces: integer (nullable = true)
 |-- total_of_special_requests: integer (nullable = true)
 |-- arrival_year: integer (nullable = true)
 |-- arrival_month: integer (nullable = true)
 |-- a

In [9]:
df.show(10, truncate=False)

26/06/11 10:35:10 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----+---------+-----------------------+--------------------+------------+------+--------+------+------------+---------------+--------------------+------+-----------------+----------------------+------------------------------+-----+-------+---------------------------+-------------------------+------------+-------------+-----------+----------------+----------+------------------+-------------+-------------------+------------------------+------------------------+--------------------+--------------------------+
|label|lead_time|stays_in_weekend_nights|stays_in_week_nights|total_nights|adults|children|babies|total_guests|booking_changes|days_in_waiting_list|adr   |is_repeated_guest|previous_cancellations|previous_bookings_not_canceled|agent|company|required_car_parking_spaces|total_of_special_requests|arrival_year|arrival_month|arrival_day|hotel_name_index|meal_index|deposit_type_index|country_index|customer_type_index|reserved_room_type_index|assigned_room_type_index|market_segment_index|

In [13]:
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

+-----+-----------+---------+-----------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------+--------+------+----+-------+--------------+--------------------+-----------------+----------------------+------------------------------+------------------+------------------+---------------+------------+-----+-------+--------------------+-------------+---+---------------------------+-------------------------+------------------+-----------------------+
|hotel|is_canceled|lead_time|arrival_date_year|arrival_date_month|arrival_date_week_number|arrival_date_day_of_month|stays_in_weekend_nights|stays_in_week_nights|adults|children|babies|meal|country|market_segment|distribution_channel|is_repeated_guest|previous_cancellations|previous_bookings_not_canceled|reserved_room_type|assigned_room_type|booking_changes|deposit_type|agent|company|days_in_waiting_list|customer_type|adr|required_car_parking_spaces|total_of_special_reque

In [ ]:
total_rows = df.count()

distinct_rows = df.distinct().count()

duplicate_rows_count = total_rows - distinct_rows
print(f"Số dòng bị lặp: {duplicate_rows_count}")

Số dòng bị lặp: 31994


Kiểm tra số dòng lặp chính xác

In [26]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

windowSpec = Window.partitionBy(df.columns).orderBy(df.columns[0])

df_with_row_num = df.withColumn("row_num", row_number().over(windowSpec))


two_duplicate_rows = df_with_row_num.filter(
    col("row_num").isin([1, 2])
).join(

    duplicates.select(df.columns), 
    on=df.columns, 
    how="inner"
)

two_duplicate_rows.drop("row_num").show()

+----------+-----------+---------+-----------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------+--------+------+----+-------+--------------+--------------------+-----------------+----------------------+------------------------------+------------------+------------------+---------------+------------+-----+-------+--------------------+---------------+-----+---------------------------+-------------------------+------------------+-----------------------+
|     hotel|is_canceled|lead_time|arrival_date_year|arrival_date_month|arrival_date_week_number|arrival_date_day_of_month|stays_in_weekend_nights|stays_in_week_nights|adults|children|babies|meal|country|market_segment|distribution_channel|is_repeated_guest|previous_cancellations|previous_bookings_not_canceled|reserved_room_type|assigned_room_type|booking_changes|deposit_type|agent|company|days_in_waiting_list|  customer_type|  adr|required_car_parking_spaces|tota

In [27]:
df = df.dropDuplicates()

In [29]:
df.groupBy("Hotel").count().orderBy(desc("count")).show()

+------------+-----+
|       Hotel|count|
+------------+-----+
|  City Hotel|53428|
|Resort Hotel|33968|
+------------+-----+



In [30]:
df.groupBy("arrival_date_month").count().orderBy(desc("count")).show()   

+------------------+-----+
|arrival_date_month|count|
+------------------+-----+
|            August|11257|
|              July|10057|
|               May| 8355|
|             April| 7908|
|              June| 7765|
|             March| 7513|
|           October| 6934|
|         September| 6690|
|          February| 6098|
|          December| 5131|
|          November| 4995|
|           January| 4693|
+------------------+-----+



In [31]:
df.groupBy("children").count().orderBy(desc("count")).show()   

+--------+-----+
|children|count|
+--------+-----+
|       0|79028|
|       1| 4695|
|       2| 3593|
|       3|   75|
|      NA|    4|
|      10|    1|
+--------+-----+



In [32]:
df.groupBy("babies").count().orderBy(desc("count")).show()   

+------+-----+
|babies|count|
+------+-----+
|     0|86482|
|     1|  897|
|     2|   15|
|    10|    1|
|     9|    1|
+------+-----+



In [36]:
df.groupBy("market_segment").count().orderBy(desc("count")).show()   

+--------------+-----+
|market_segment|count|
+--------------+-----+
|     Online TA|51618|
| Offline TA/TO|13889|
|        Direct|11804|
|        Groups| 4942|
|     Corporate| 4212|
| Complementary|  702|
|      Aviation|  227|
|     Undefined|    2|
+--------------+-----+



In [37]:
df.groupBy("distribution_channel").count().orderBy(desc("count")).show()   

+--------------------+-----+
|distribution_channel|count|
+--------------------+-----+
|               TA/TO|69141|
|              Direct|12988|
|           Corporate| 5081|
|                 GDS|  181|
|           Undefined|    5|
+--------------------+-----+



In [38]:
df.groupBy("reserved_room_type").count().orderBy(desc("count")).show() 

+------------------+-----+
|reserved_room_type|count|
+------------------+-----+
|                 A|56552|
|                 D|17398|
|                 E| 6049|
|                 F| 2823|
|                 G| 2052|
|                 B|  999|
|                 C|  915|
|                 H|  596|
|                 L|    6|
|                 P|    6|
+------------------+-----+



In [39]:
df.groupBy("assigned_room_type").count().orderBy(desc("count")).show() 

+------------------+-----+
|assigned_room_type|count|
+------------------+-----+
|                 A|46313|
|                 D|22432|
|                 E| 7195|
|                 F| 3627|
|                 G| 2498|
|                 C| 2165|
|                 B| 1820|
|                 H|  706|
|                 I|  357|
|                 K|  276|
|                 P|    6|
|                 L|    1|
+------------------+-----+



In [41]:
df.groupBy("deposit_type").count().orderBy(desc("count")).show() 

+------------+-----+
|deposit_type|count|
+------------+-----+
|  No Deposit|86251|
|  Non Refund| 1038|
|  Refundable|  107|
+------------+-----+



In [46]:
df.groupBy("reservation_status").count().orderBy(desc("count")).show() 

+------------------+-----+
|reservation_status|count|
+------------------+-----+
|         Check-Out|63371|
|          Canceled|23011|
|           No-Show| 1014|
+------------------+-----+



In [47]:
df.groupBy("customer_type").count().orderBy(desc("count")).show() 

+---------------+-----+
|  customer_type|count|
+---------------+-----+
|      Transient|71986|
|Transient-Party|11727|
|       Contract| 3139|
|          Group|  544|
+---------------+-----+



In [50]:
numeric_cols = [
    field.name
    for field in df.schema.fields
    if isinstance(field.dataType, NumericType)
]

print(numeric_cols)

['is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'babies', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'booking_changes', 'days_in_waiting_list', 'adr', 'required_car_parking_spaces', 'total_of_special_requests']


In [52]:
df.select(numeric_cols).describe().show()

+-------+-------------------+-----------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------------------+--------------------+-------------------+----------------------+------------------------------+-------------------+--------------------+------------------+---------------------------+-------------------------+
|summary|        is_canceled|        lead_time| arrival_date_year|arrival_date_week_number|arrival_date_day_of_month|stays_in_weekend_nights|stays_in_week_nights|            adults|              babies|  is_repeated_guest|previous_cancellations|previous_bookings_not_canceled|    booking_changes|days_in_waiting_list|               adr|required_car_parking_spaces|total_of_special_requests|
+-------+-------------------+-----------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------------------+--------------------+----------------

Missing value

In [53]:
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+-----+-----------+---------+-----------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------+--------+------+----+-------+--------------+--------------------+-----------------+----------------------+------------------------------+------------------+------------------+---------------+------------+-----+-------+--------------------+-------------+---+---------------------------+-------------------------+------------------+-----------------------+
|hotel|is_canceled|lead_time|arrival_date_year|arrival_date_month|arrival_date_week_number|arrival_date_day_of_month|stays_in_weekend_nights|stays_in_week_nights|adults|children|babies|meal|country|market_segment|distribution_channel|is_repeated_guest|previous_cancellations|previous_bookings_not_canceled|reserved_room_type|assigned_room_type|booking_changes|deposit_type|agent|company|days_in_waiting_list|customer_type|adr|required_car_parking_spaces|total_of_special_reque

26/06/10 14:04:59 ERROR TaskSchedulerImpl: Lost executor 1 on 192.168.252.7: worker lost: Not receiving heartbeat for 60 seconds


In [4]:
spark.stop()